# Notebook 1 / R1 - EfficientNet-B4 Teacher

Notebook ini adalah template R1 untuk melatih EfficientNet-B4 sebagai teacher utama. Pilot R1 sudah lolos dan final seed 42 sudah berhasil, sehingga default notebook sekarang diarahkan ke final seed berikutnya. Split tidak dibuat ulang di notebook ini; semua train/validation/test row dibaca dari artefak R0.

## Rules

- Gunakan `split_manifest_seed_*.csv` dari R0.
- Train set hanya untuk training.
- Validation set untuk checkpoint selection.
- Default run sekarang adalah final: 100 epoch, early stopping nonaktif, test set dievaluasi untuk final reporting.
- Jika ingin mencoba pilot lain, ubah `RUN_PHASE="pilot"`.
- Pilot output disimpan di `final_research_kd/runs/pilots/R1/<setup_id>/`.
- Final output disimpan di `final_research_kd/runs/final/R1/seed_<seed>/`.
- Untuk final 5-seed run, pertahankan `RUN_PHASE="final"`, lalu ubah `SEED` dan rerun notebook ini untuk setiap seed.
- Status saat ini: seed `42` sudah selesai; lanjutkan seed `123`, `777`, `2026`, dan `3407`.

In [ ]:
# ============================================================
# 1. Imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import copy
import json
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=UserWarning)

print("Imports ready")

In [ ]:
# ============================================================
# 2. Configuration
# ============================================================

class Config:
    EXPERIMENT_ID = "R1"
    EXPERIMENT_NAME = "R1 EfficientNet-B4 Teacher"

    PLATFORM = "Kaggle Notebooks"
    FRAMEWORK = "PyTorch"
    USE_AMP = True
    MULTI_GPU = False

    # Final seed schedule: 42 done; continue with 123, 777, 2026, 3407.
    SEED = 123
    RUN_PHASE = "final"  # "pilot" for smoke test/tuning, "final" for 5-seed reporting

    DATASET_NAME = "TrashNet"
    DATASET_DIR = Path(os.environ.get(
        "TRASHNET_DATASET_DIR",
        "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
    ))

    R0_DIR_CANDIDATES = [
        Path(os.environ.get("R0_DATA_PROTOCOL_DIR", "")),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup/final_research"),
        Path("/kaggle/input/notebook0-r0-final-data-protocol-setup"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup/final_research"),
        Path("/kaggle/input/notebook0_r0_final_data_protocol_setup"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-final-data-protocol-setup/r0_data_protocol"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-final-data-protocol-setup/final_research/r0_data_protocol"),
        Path("/kaggle/input/notebooks/hamzapratama/notebook0-r0-final-data-protocol-setup"),
        Path("/kaggle/working/final_research/r0_data_protocol"),
        Path("/kaggle/input/r0-kaggle-final/r0_data_protocol"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research/runs/r0_kaggle_final/r0_data_protocol"),
        Path("/kaggle/input/thesis-kd-trashnet/final_research_kd/runs/r0_kaggle_final/final_research/r0_data_protocol"),
        Path.cwd() / "final_research" / "runs" / "r0_kaggle_final" / "r0_data_protocol",
        Path.cwd() / "final_research_kd" / "runs" / "r0_kaggle_final" / "final_research" / "r0_data_protocol",
        Path.cwd() / "runs" / "r0_kaggle_final" / "r0_data_protocol",
    ]

    DEFAULT_OUTPUT_ROOT = (
        Path("/kaggle/working/final_research_kd/runs")
        if Path("/kaggle/working").exists()
        else Path.cwd() / "final_research_kd" / "runs"
    )
    OUTPUT_ROOT = Path(os.environ.get("R1_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT)))

    MODEL_NAME = "EfficientNet-B4"
    MODEL_TIMM_NAME = "efficientnet_b4"
    PRETRAINED = True

    IMG_SIZE = 380
    FINAL_EPOCHS = 100
    PILOT_EPOCHS = 10
    EPOCHS = PILOT_EPOCHS if RUN_PHASE == "pilot" else FINAL_EPOCHS
    BATCH_SIZE = 8
    NUM_WORKERS = 2

    LR = 0.05
    MOMENTUM = 0.9
    WEIGHT_DECAY = 1e-4
    SCHEDULER = "CosineAnnealingLR"
    CHECKPOINT_METRIC = "best_val_accuracy"
    EARLY_STOPPING = RUN_PHASE == "pilot"
    EARLY_STOPPING_MONITOR = "val_loss"
    PATIENCE = 5

    EVALUATE_TEST = RUN_PHASE == "final"


def float_tag(value):
    return str(value).replace(".", "p")


cfg = Config()
if cfg.RUN_PHASE == "pilot":
    cfg.SETUP_ID = (
        f"r1_pilot_b4_s{cfg.SEED}_e{cfg.EPOCHS}_"
        f"es-{cfg.EARLY_STOPPING_MONITOR.replace('_', '')}-p{cfg.PATIENCE}_"
        f"img{cfg.IMG_SIZE}_lr{float_tag(cfg.LR)}_bs{cfg.BATCH_SIZE}"
    )
else:
    cfg.SETUP_ID = f"r1_final_b4_s{cfg.SEED}_e{cfg.EPOCHS}_img{cfg.IMG_SIZE}_lr{float_tag(cfg.LR)}_bs{cfg.BATCH_SIZE}"

if cfg.RUN_PHASE == "pilot":
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "pilots" / cfg.EXPERIMENT_ID / cfg.SETUP_ID
elif cfg.RUN_PHASE == "final":
    cfg.OUTPUT_DIR = cfg.OUTPUT_ROOT / "final" / cfg.EXPERIMENT_ID / f"seed_{cfg.SEED}"
else:
    raise ValueError(f"Unsupported RUN_PHASE: {cfg.RUN_PHASE}")
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Experiment : {cfg.EXPERIMENT_NAME}")
print(f"Run phase  : {cfg.RUN_PHASE}")
print(f"Seed       : {cfg.SEED}")
print(f"Setup ID   : {cfg.SETUP_ID}")
print(f"Epochs     : {cfg.EPOCHS}")
print(f"Early stop : {cfg.EARLY_STOPPING} (patience={cfg.PATIENCE})")
print(f"Eval test  : {cfg.EVALUATE_TEST}")
print(f"Dataset dir: {cfg.DATASET_DIR}")
print(f"Output dir : {cfg.OUTPUT_DIR}")

In [ ]:
# ============================================================
# 3. Reproducibility and Device
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"VRAM GB: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")

## 1. Load R0 Artifacts

Cell ini wajib sukses. Kalau R0 artifact tidak ditemukan, jangan lanjut training.

In [ ]:
# ============================================================
# 4. Resolve R0 Artifacts
# ============================================================

def has_r0_artifacts(path: Path) -> bool:
    return (
        (path / "class_mapping.json").exists()
        and any(path.glob("split_manifest_seed_*.csv"))
    )


def candidate_variants(candidate: Path):
    yield candidate
    yield candidate / "r0_data_protocol"
    yield candidate / "final_research" / "r0_data_protocol"


def discover_r0_dirs():
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
    discovered = []
    for root in roots:
        if not root.exists():
            continue
        try:
            discovered.extend([path for path in root.rglob("r0_data_protocol") if path.is_dir()])
        except Exception as exc:
            print(f"Skipping R0 discovery under {root}: {exc}")
    return discovered


def resolve_existing_dir(candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if str(candidate) in {"", "."}:
            continue
        for variant in candidate_variants(candidate):
            if variant.exists() and has_r0_artifacts(variant):
                return variant

    discovered = discover_r0_dirs()
    valid_discovered = [path for path in discovered if has_r0_artifacts(path)]
    if valid_discovered:
        print("Auto-discovered R0 candidates:")
        for path in valid_discovered:
            print(f"- {path}")
        return valid_discovered[0]

    print("Checked R0 candidates:")
    for candidate in candidates:
        print(f"- {candidate}")
    print("Discovered r0_data_protocol dirs:")
    for path in discovered:
        print(f"- {path}")
    raise FileNotFoundError("R0 data protocol directory not found. Set R0_DATA_PROTOCOL_DIR.")


R0_DIR = resolve_existing_dir(cfg.R0_DIR_CANDIDATES)
CLASS_MAPPING_PATH = R0_DIR / "class_mapping.json"
MANIFEST_PATH = R0_DIR / f"split_manifest_seed_{cfg.SEED}.csv"
EXCLUDED_DUPLICATES_PATH = R0_DIR / "excluded_duplicate_conflicts.csv"

for path in [CLASS_MAPPING_PATH, MANIFEST_PATH, EXCLUDED_DUPLICATES_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required R0 artifact missing: {path}")

with CLASS_MAPPING_PATH.open("r", encoding="utf-8") as handle:
    class_mapping = json.load(handle)

manifest_df = pd.read_csv(MANIFEST_PATH)
excluded_df = pd.read_csv(EXCLUDED_DUPLICATES_PATH)

CLASS_NAMES = class_mapping["class_names"]
CLASS_TO_IDX = class_mapping["class_to_idx"]
NUM_CLASSES = len(CLASS_NAMES)

print(f"R0 dir       : {R0_DIR}")
print(f"Manifest     : {MANIFEST_PATH.name}")
print(f"Rows         : {len(manifest_df)}")
print(f"Classes      : {CLASS_NAMES}")
print(f"Excluded dup : {len(excluded_df)} rows")
display(manifest_df.groupby(["split", "label", "class_id"], as_index=False).size())

In [ ]:
# ============================================================
# 5. Manifest Validation
# ============================================================

required_columns = {"sample_id", "image_path", "relative_path", "label", "class_id", "split", "seed", "sha256"}
missing_columns = required_columns - set(manifest_df.columns)
if missing_columns:
    raise ValueError(f"Manifest missing columns: {sorted(missing_columns)}")

if set(manifest_df["seed"].unique()) != {cfg.SEED}:
    raise ValueError(f"Manifest seed mismatch. Expected only {cfg.SEED}.")

if set(manifest_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError("Manifest must contain train, val, and test splits.")

excluded_ids = set(excluded_df.get("sample_id", []))
if excluded_ids & set(manifest_df["sample_id"]):
    raise AssertionError("Excluded duplicate-conflict samples are present in this manifest.")

for class_name, class_id in CLASS_TO_IDX.items():
    rows = manifest_df[manifest_df["label"] == class_name]
    if rows.empty:
        raise AssertionError(f"Missing class in manifest: {class_name}")
    if set(rows["class_id"].unique()) != {class_id}:
        raise AssertionError(f"Class id mismatch for {class_name}")

print("Manifest validation passed")

## 2. Dataset and DataLoader

In [ ]:
# ============================================================
# 6. Transforms
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def coarse_dropout(img_size: int):
    min_h = int(img_size * 0.05)
    max_h = int(img_size * 0.20)
    try:
        return A.CoarseDropout(
            num_holes_range=(1, 1),
            hole_height_range=(min_h, max_h),
            hole_width_range=(min_h, max_h),
            fill=0,
            p=0.5,
        )
    except TypeError:
        return A.CoarseDropout(
            max_holes=1,
            min_height=min_h,
            max_height=max_h,
            min_width=min_h,
            max_width=max_h,
            fill_value=0,
            p=0.5,
        )


train_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    coarse_dropout(cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print("Transforms ready")

In [ ]:
# ============================================================
# 7. Manifest Dataset
# ============================================================

class TrashNetManifestDataset(Dataset):
    def __init__(self, df: pd.DataFrame, dataset_dir: Path, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.dataset_dir = Path(dataset_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def resolve_path(self, row) -> Path:
        absolute_path = Path(row["image_path"])
        if absolute_path.exists():
            return absolute_path
        fallback_path = self.dataset_dir / row["relative_path"]
        if fallback_path.exists():
            return fallback_path
        raise FileNotFoundError(f"Image not found: {absolute_path} or {fallback_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.resolve_path(row)
        image = Image.open(image_path).convert("RGB")
        image = np.array(image)
        if self.transform is not None:
            image = self.transform(image=image)["image"]
        return {
            "image": image,
            "label": int(row["class_id"]),
            "sample_id": row["sample_id"],
            "relative_path": row["relative_path"],
        }


train_df = manifest_df[manifest_df["split"] == "train"].copy()
val_df = manifest_df[manifest_df["split"] == "val"].copy()
test_df = manifest_df[manifest_df["split"] == "test"].copy()

train_dataset = TrashNetManifestDataset(train_df, cfg.DATASET_DIR, train_transform)
val_dataset = TrashNetManifestDataset(val_df, cfg.DATASET_DIR, eval_transform)
test_dataset = TrashNetManifestDataset(test_df, cfg.DATASET_DIR, eval_transform)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# ============================================================
# 8. DataLoaders and Distribution
# ============================================================

def worker_init_fn(worker_id):
    worker_seed = cfg.SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


generator = torch.Generator()
generator.manual_seed(cfg.SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
    generator=generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=True,
)

split_counts = manifest_df.groupby(["split", "label", "class_id"], as_index=False).size()
display(split_counts)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

## 3. Model and Training Components

In [ ]:
# ============================================================
# 9. Model
# ============================================================

def create_teacher_model(pretrained: bool):
    return timm.create_model(
        cfg.MODEL_TIMM_NAME,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
    )


model = create_teacher_model(pretrained=cfg.PRETRAINED).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model        : {cfg.MODEL_NAME}")
print(f"timm name    : {cfg.MODEL_TIMM_NAME}")
print(f"Pretrained   : {cfg.PRETRAINED}")
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable_params:,}")

In [ ]:
# ============================================================
# 10. Loss, Optimizer, Scheduler
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=cfg.LR,
    momentum=cfg.MOMENTUM,
    weight_decay=cfg.WEIGHT_DECAY,
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
scaler = GradScaler(enabled=cfg.USE_AMP)

print("Training components ready")

In [ ]:
# ============================================================
# 11. Train / Validate Helpers
# ============================================================

def run_one_train_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total


@torch.no_grad()
def run_eval_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

## 4. Training

In [ ]:
# ============================================================
# 12. Main Training Loop
# ============================================================

history = []
best_val_acc = -1.0
best_epoch = 0
best_model_state = None
best_monitor_value = np.inf if cfg.EARLY_STOPPING_MONITOR == "val_loss" else -np.inf
epochs_without_improvement = 0
total_start = time.time()

print(f"Starting {cfg.EXPERIMENT_NAME} | seed={cfg.SEED}")
for epoch in range(1, cfg.EPOCHS + 1):
    epoch_start = time.time()
    current_lr = optimizer.param_groups[0]["lr"]

    train_loss, train_acc = run_one_train_epoch(model, train_loader)
    val_loss, val_acc = run_eval_epoch(model, val_loader)
    scheduler.step()

    epoch_time = time.time() - epoch_start
    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())

    if cfg.EARLY_STOPPING_MONITOR == "val_loss":
        monitor_value = val_loss
        monitor_improved = monitor_value < best_monitor_value - 1e-8
    elif cfg.EARLY_STOPPING_MONITOR == "val_acc":
        monitor_value = val_acc
        monitor_improved = monitor_value > best_monitor_value + 1e-8
    else:
        raise ValueError(f"Unsupported EARLY_STOPPING_MONITOR: {cfg.EARLY_STOPPING_MONITOR}")

    if monitor_improved:
        best_monitor_value = monitor_value
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr,
        "epoch_time_sec": epoch_time,
        "is_best": is_best,
        "early_stop_monitor": cfg.EARLY_STOPPING_MONITOR,
        "early_stop_value": monitor_value,
        "early_stop_improved": monitor_improved,
        "epochs_without_improvement": epochs_without_improvement,
    })

    marker = " BEST" if is_best else ""
    early_stop_status = f" | es_wait={epochs_without_improvement}/{cfg.PATIENCE}" if cfg.EARLY_STOPPING else ""
    print(
        f"Epoch {epoch:03d}/{cfg.EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.6f} time={epoch_time:.1f}s{early_stop_status}{marker}"
    )

    if cfg.EARLY_STOPPING and epochs_without_improvement >= cfg.PATIENCE:
        print(
            f"Early stopping triggered at epoch {epoch} "
            f"({cfg.EARLY_STOPPING_MONITOR} did not improve for {cfg.PATIENCE} epochs)."
        )
        break

total_train_time = time.time() - total_start
history_df = pd.DataFrame(history)

print("Training complete")
print(f"Best epoch: {best_epoch}")
print(f"Best val acc: {best_val_acc:.6f}")
print(f"Total minutes: {total_train_time / 60:.1f}")

## 5. Evaluation and Prediction CSV

In [ ]:
# ============================================================
# 13. Evaluation Helpers
# ============================================================

@torch.no_grad()
def collect_predictions(model, loader, split_name: str) -> pd.DataFrame:
    model.eval()
    rows = []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].cpu().numpy()
        with autocast(enabled=cfg.USE_AMP):
            logits = model(images)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        for i in range(len(labels)):
            row = {
                "sample_id": batch["sample_id"][i],
                "image_path": batch["relative_path"][i],
                "label": CLASS_NAMES[int(labels[i])],
                "label_id": int(labels[i]),
                "prediction": CLASS_NAMES[int(preds[i])],
                "prediction_id": int(preds[i]),
                "seed": cfg.SEED,
                "model_id": cfg.EXPERIMENT_ID,
                "split": split_name,
            }
            for class_idx, class_name in enumerate(CLASS_NAMES):
                row[f"prob_{class_name}"] = float(probs[i, class_idx])
            rows.append(row)
    return pd.DataFrame(rows)


def compute_metrics(pred_df: pd.DataFrame) -> dict:
    y_true = pred_df["label_id"].to_numpy()
    y_pred = pred_df["prediction_id"].to_numpy()
    prob_cols = [f"prob_{class_name}" for class_name in CLASS_NAMES]
    y_prob = pred_df[prob_cols].to_numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    try:
        auc_macro_ovr = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except ValueError:
        auc_macro_ovr = np.nan

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
        "auc_macro_ovr": auc_macro_ovr,
    }


def save_confusion_matrix(pred_df: pd.DataFrame, split_name: str):
    cm = confusion_matrix(pred_df["label_id"], pred_df["prediction_id"], labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(NUM_CLASSES))
    ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{cfg.EXPERIMENT_ID} Confusion Matrix - {split_name}")
    for r in range(NUM_CLASSES):
        for c in range(NUM_CLASSES):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", color="black")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    path = cfg.OUTPUT_DIR / f"confusion_matrix_{split_name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path

In [ ]:
# ============================================================
# 14. Load Best State and Evaluate
# ============================================================

model.load_state_dict(best_model_state)

val_predictions_df = collect_predictions(model, val_loader, "val")
val_metrics = compute_metrics(val_predictions_df)

test_predictions_df = pd.DataFrame()
test_metrics = None
if cfg.EVALUATE_TEST and cfg.RUN_PHASE == "final":
    test_predictions_df = collect_predictions(model, test_loader, "test")
    test_metrics = compute_metrics(test_predictions_df)
else:
    print("Independent test evaluation skipped. Set RUN_PHASE='final' and EVALUATE_TEST=True to enable it.")

print("Validation metrics:")
display(pd.DataFrame([val_metrics]))
if test_metrics is not None:
    print("Test metrics:")
    display(pd.DataFrame([test_metrics]))

val_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_val.csv"
val_predictions_df.to_csv(val_pred_path, index=False)
print(f"Saved: {val_pred_path}")

test_pred_path = None
if not test_predictions_df.empty:
    test_pred_path = cfg.OUTPUT_DIR / f"predictions_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_test.csv"
    test_predictions_df.to_csv(test_pred_path, index=False)
    print(f"Saved: {test_pred_path}")

save_confusion_matrix(val_predictions_df, "val")
if not test_predictions_df.empty:
    save_confusion_matrix(test_predictions_df, "test")

## 6. Save Artifacts

In [ ]:
# ============================================================
# 15. Training Curves and History
# ============================================================

history_path = cfg.OUTPUT_DIR / f"training_history_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}.csv"
history_df.to_csv(history_path, index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_acc"], label="train")
axes[1].plot(history_df["epoch"], history_df["val_acc"], label="val")
axes[1].axvline(best_epoch, color="red", linestyle="--", alpha=0.6)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(history_df["epoch"], history_df["lr"], color="green")
axes[2].set_title("Learning Rate")
axes[2].set_xlabel("Epoch")
axes[2].grid(alpha=0.3)

fig.suptitle(f"{cfg.EXPERIMENT_ID} EfficientNet-B4 Teacher - seed {cfg.SEED}")
fig.tight_layout()
curve_path = cfg.OUTPUT_DIR / f"training_curves_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}.png"
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {history_path}")
print(f"Saved: {curve_path}")

In [ ]:
# ============================================================
# 16. Checkpoint, Metrics, Artifact Manifest
# ============================================================

metrics_rows = []
metrics_rows.append({"split": "val", **val_metrics})
if test_metrics is not None:
    metrics_rows.append({"split": "test", **test_metrics})
metrics_df = pd.DataFrame(metrics_rows)
metrics_path = cfg.OUTPUT_DIR / f"metrics_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}.csv"
metrics_df.to_csv(metrics_path, index=False)

config_dict = {
    "experiment_id": cfg.EXPERIMENT_ID,
    "experiment_name": cfg.EXPERIMENT_NAME,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "setup_id": cfg.SETUP_ID,
    "output_root": str(cfg.OUTPUT_ROOT),
    "output_dir": str(cfg.OUTPUT_DIR),
    "dataset_dir": str(cfg.DATASET_DIR),
    "r0_dir": str(R0_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "model_name": cfg.MODEL_NAME,
    "model_timm_name": cfg.MODEL_TIMM_NAME,
    "pretrained": cfg.PRETRAINED,
    "img_size": cfg.IMG_SIZE,
    "epochs": cfg.EPOCHS,
    "final_epochs": cfg.FINAL_EPOCHS,
    "pilot_epochs": cfg.PILOT_EPOCHS,
    "batch_size": cfg.BATCH_SIZE,
    "optimizer": "SGD",
    "lr": cfg.LR,
    "momentum": cfg.MOMENTUM,
    "weight_decay": cfg.WEIGHT_DECAY,
    "scheduler": cfg.SCHEDULER,
    "use_amp": cfg.USE_AMP,
    "early_stopping": cfg.EARLY_STOPPING,
    "early_stopping_monitor": cfg.EARLY_STOPPING_MONITOR,
    "patience": cfg.PATIENCE,
    "checkpoint_metric": cfg.CHECKPOINT_METRIC,
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "train_size": int(len(train_df)),
    "val_size": int(len(val_df)),
    "test_size": int(len(test_df)),
    "total_params": int(total_params),
    "trainable_params": int(trainable_params),
}

checkpoint = {
    "model_state_dict": best_model_state,
    "best_epoch": best_epoch,
    "best_val_acc": float(best_val_acc),
    "val_metrics": val_metrics,
    "test_metrics": test_metrics,
    "config": config_dict,
    "class_mapping": class_mapping,
}

checkpoint_path = cfg.OUTPUT_DIR / f"efficientnet_b4_teacher_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}_best.pth"
torch.save(checkpoint, checkpoint_path)

config_path = cfg.OUTPUT_DIR / f"config_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}.json"
config_path.write_text(json.dumps(config_dict, indent=2), encoding="utf-8")

artifact_manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_id": cfg.EXPERIMENT_ID,
    "seed": cfg.SEED,
    "run_phase": cfg.RUN_PHASE,
    "setup_id": cfg.SETUP_ID,
    "output_dir": str(cfg.OUTPUT_DIR),
    "artifacts": {
        "checkpoint": str(checkpoint_path),
        "config": str(config_path),
        "history": str(history_path),
        "metrics": str(metrics_path),
        "val_predictions": str(val_pred_path),
        "test_predictions": str(test_pred_path) if test_pred_path else None,
        "training_curves": str(curve_path),
    },
}
artifact_manifest_path = cfg.OUTPUT_DIR / f"artifact_manifest_{cfg.EXPERIMENT_ID.lower()}_{cfg.RUN_PHASE}_seed_{cfg.SEED}.json"
artifact_manifest_path.write_text(json.dumps(artifact_manifest, indent=2), encoding="utf-8")

print(f"Saved checkpoint: {checkpoint_path}")
print(f"Saved metrics   : {metrics_path}")
print(f"Saved manifest  : {artifact_manifest_path}")
display(metrics_df)

In [ ]:
# ============================================================
# 17. Checkpoint Verification
# ============================================================

# PyTorch 2.6 defaults torch.load(weights_only=True). This checkpoint includes
# trusted metadata saved by this notebook, so load it with weights_only=False.
loaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
verify_model = create_teacher_model(pretrained=False)
verify_model.load_state_dict(loaded["model_state_dict"])

print("Checkpoint verification passed")
print(f"Best epoch   : {loaded['best_epoch']}")
print(f"Best val acc : {loaded['best_val_acc']:.6f}")

del loaded, verify_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Output Artifacts

Per seed, notebook ini menghasilkan checkpoint teacher, history CSV, metrics CSV, prediction CSV untuk validation/test, confusion matrix, config JSON, dan artifact manifest. Checkpoint R1 ini dipakai sebagai teacher untuk direct KD dan Focus-RCNet teacher-assistant selection.

In [ ]:
print("R1 complete")
print(f"Seed: {cfg.SEED}")
print(f"Best epoch: {best_epoch}")
print(f"Best val accuracy: {best_val_acc:.6f}")
if test_metrics is not None:
    print(f"Test accuracy: {test_metrics['accuracy']:.6f}")
print(f"Output dir: {cfg.OUTPUT_DIR}")